# 02 · Aerial greenery per road segment
Green area and share inside a 10 m strip on each side of every road (one merged strip per `osmid`, so overlapping edges count once), per 2 km aerial tile in parallel, summed per road.

**Inputs:** `data/raw_data/road_network/London-roads/edges.shp`; `data/raw_data/green_cover_tiles/ndvi_landcover_merge_*.shp` (not included; see the README).
**Intermediates:** `data/greenery_metrics/SAT_greenery_per_segment_tiles/` (`road_buffers.gpkg` and one CSV per tile; finished tiles are skipped on a rerun).
**Output:** `data/greenery_metrics/SAT_greenery_per_segment.csv`: `osmid, sat_green_area, sat_green_pct, segment_tot_area`.
**Runtime:** about 30 minutes on 8 cores. **Logic:** `02_SAT_greenery_per_segment_workers.py`, which also defines the road-centrality weight used by 03 and 04.

In [ ]:
import importlib
import os
import sys
from pathlib import Path

from joblib import Parallel, delayed

sys.path.insert(0, str(Path.cwd().resolve()))
gpsw = importlib.import_module('02_SAT_greenery_per_segment_workers')
print(f'roads: {gpsw.ROADS_PATH} | tiles: {gpsw.NDVI_TILES_DIR}')

## 1. Road strips and tiles in parallel

In [ ]:
N_JOBS = os.cpu_count()   # lower if memory is tight
gpsw.build_road_buffers()
tiles = gpsw.tile_paths()
todo = [t for t in tiles if not gpsw.tile_out(t).exists()]
print(f'{len(tiles) - len(todo)}/{len(tiles)} tiles done; running {len(todo)}')
Parallel(n_jobs=N_JOBS, verbose=5)(delayed(gpsw.process_tile)(t) for t in todo);

## 2. Combine the tiles → SAVE

In [ ]:
seg = gpsw.merge_tiles()
print(f'wrote {len(seg):,} osmids → {gpsw.write_csv(seg)}')
seg.head()

## Validation

In [ ]:
assert list(seg.columns) == gpsw.OUT_COLUMNS
assert seg['osmid'].is_unique and seg['sat_green_pct'].between(0, 1).all() and (seg['segment_tot_area'] > 0).all()
print(f"osmids: {len(seg):,} | no green: {(seg['sat_green_area'] == 0).sum():,} | median share: {seg['sat_green_pct'].median():.4f}")